# OddsJam market coverage (ATP vs WTA)

Goal: measure fixture-level market coverage for the markets in `injestion.oddsjam.fetch.odds.DEFAULT_MARKETS`.

Coverage definition used here:
- denominator = distinct fixtures in the odds table for a given tour (ATP/WTA)
- numerator = fixtures with at least one non-`no_odds` row for that market
- sportsbook is ignored (we dedupe at fixture + market)
- market matching uses `market_id` first, with fallback to `market`


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from google.cloud import bigquery


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "injestion").exists() and (candidate / ".env").exists():
            return candidate
    raise RuntimeError("Could not find project root containing `injestion/` and `.env`.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from injestion.core.env import load_env
from injestion.oddsjam.fetch.odds import DEFAULT_MARKETS

load_env()

ODDS_TABLE_ID = os.environ["BIGQUERY_ODDSJAM_ODDS_TABLE_ID"]
FIXTURES_TABLE_ID = os.environ["BIGQUERY_ODDSJAM_FIXTURES_TABLE_ID"]
MARKETS = list(DEFAULT_MARKETS)

START_DATE = "2025-06-20"
END_DATE = "2026-06-20"

client = bigquery.Client()

print(f"Project root: {PROJECT_ROOT}")
print(f"Odds table: {ODDS_TABLE_ID}")
print(f"Fixtures table: {FIXTURES_TABLE_ID}")
print(f"Date window: {START_DATE} to {END_DATE}")
print(f"Markets in scope: {len(MARKETS)}")


Project root: /Users/matt.holden/Projects/tennis-origination
Odds table: prizepicksanalytics.originations_tennis.oj_odds
Fixtures table: prizepicksanalytics.originations_tennis.oj_fixtures
Date window: 2025-06-20 to 2026-06-20
Markets in scope: 33


In [2]:
query = f"""
WITH date_filtered_fixtures AS (
  SELECT DISTINCT id AS fixture_id
  FROM `{FIXTURES_TABLE_ID}`
  WHERE id IS NOT NULL
    AND DATE(start_date) BETWEEN @start_date AND @end_date
    AND UPPER(IFNULL(status, '')) != 'CANCELLED'
),
fixture_universe AS (
  SELECT DISTINCT
    o.fixture_id,
    CASE
      WHEN REGEXP_CONTAINS(UPPER(o.league_name), r'(^|[^A-Z])ATP([^A-Z]|$)') THEN 'ATP'
      WHEN REGEXP_CONTAINS(UPPER(o.league_name), r'(^|[^A-Z])WTA([^A-Z]|$)') THEN 'WTA'
      ELSE NULL
    END AS tour
  FROM `{ODDS_TABLE_ID}` o
  INNER JOIN date_filtered_fixtures dff
    ON o.fixture_id = dff.fixture_id
  WHERE o.fixture_id IS NOT NULL
),
tour_fixtures AS (
  SELECT fixture_id, tour
  FROM fixture_universe
  WHERE tour IN ('ATP', 'WTA')
),
target_markets AS (
  SELECT LOWER(market) AS market
  FROM UNNEST(@markets) AS market
),
fixture_market_hits AS (
  SELECT DISTINCT
    o.fixture_id,
    LOWER(COALESCE(NULLIF(o.market_id, ''), NULLIF(o.market, ''))) AS market
  FROM `{ODDS_TABLE_ID}` o
  INNER JOIN date_filtered_fixtures dff
    ON o.fixture_id = dff.fixture_id
  WHERE o.fixture_id IS NOT NULL
    AND LOWER(COALESCE(NULLIF(o.market_id, ''), NULLIF(o.market, ''))) IN (
      SELECT market FROM target_markets
    )
    AND IFNULL(o.no_odds, FALSE) = FALSE
)
SELECT
  tf.tour,
  tm.market,
  COUNT(DISTINCT tf.fixture_id) AS total_fixtures,
  COUNT(DISTINCT fmh.fixture_id) AS fixtures_with_lines,
  SAFE_DIVIDE(COUNT(DISTINCT fmh.fixture_id), COUNT(DISTINCT tf.fixture_id)) AS coverage_pct
FROM tour_fixtures tf
CROSS JOIN target_markets tm
LEFT JOIN fixture_market_hits fmh
  ON tf.fixture_id = fmh.fixture_id
 AND tm.market = fmh.market
GROUP BY 1, 2
ORDER BY tf.tour, coverage_pct DESC, tm.market
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter("markets", "STRING", MARKETS),
        bigquery.ScalarQueryParameter("start_date", "DATE", START_DATE),
        bigquery.ScalarQueryParameter("end_date", "DATE", END_DATE),
    ]
)

coverage_df = client.query(query, job_config=job_config).result().to_dataframe()
coverage_df["coverage_pct"] = coverage_df["coverage_pct"].fillna(0.0)
coverage_df["coverage_pct_100"] = (coverage_df["coverage_pct"] * 100).round(2)

coverage_df


E0000 00:00:1783612136.683993 2444645 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,tour,market,total_fixtures,fixtures_with_lines,coverage_pct,coverage_pct_100
0,ATP,moneyline,3937,3925,0.996952,99.70
1,ATP,total_games,3937,3920,0.995682,99.57
2,ATP,player_games_won,3937,3916,0.994666,99.47
3,ATP,total_sets,3937,3915,0.994412,99.44
4,ATP,1st_set_total_games,3937,3912,0.993650,99.36
...,...,...,...,...,...,...
61,WTA,1st_set_game_5_total_points,6369,4,0.000628,0.06
62,WTA,1st_set_game_6_total_points,6369,3,0.000471,0.05
63,WTA,1st_set_total_points,6369,0,0.000000,0.00
64,WTA,player_first_serve_percentage,6369,0,0.000000,0.00


In [3]:
coverage_pivot = (
    coverage_df
    .pivot(index="market", columns="tour", values="coverage_pct_100")
    .fillna(0.0)
    .sort_values(by=["ATP", "WTA"], ascending=False)
)

coverage_pivot


tour,ATP,WTA
market,,
moneyline,99.70,99.21
total_games,99.57,98.95
player_games_won,99.47,97.72
total_sets,99.44,98.38
1st_set_total_games,99.36,97.94
player_sets_won,96.70,88.70
1st_set_player_games_won,94.03,81.68
total_tie_breaks,82.55,77.09
player_aces,65.81,64.14


In [4]:
# Optional: quick view of the biggest coverage gaps by tour.
coverage_gaps = (
    coverage_df
    .sort_values(["tour", "coverage_pct"], ascending=[True, True])
    [["tour", "market", "fixtures_with_lines", "total_fixtures", "coverage_pct_100"]]
)

coverage_gaps


,tour,market,fixtures_with_lines,total_fixtures,coverage_pct_100
30,ATP,1st_set_total_points,0,3937,0.00
31,ATP,player_first_serve_percentage,0,3937,0.00
32,ATP,player_service_games_lost,0,3937,0.00
29,ATP,1st_set_game_6_moneyline,3,3937,0.08
26,ATP,1st_set_game_4_total_points,6,3937,0.15
...,...,...,...,...,...
37,WTA,player_games_won,6224,6369,97.72
36,WTA,1st_set_total_games,6238,6369,97.94
35,WTA,total_sets,6266,6369,98.38
34,WTA,total_games,6302,6369,98.95


In [5]:
# Optional sanity check: does the market list hit market_id or market?
diagnostic_query = f"""
WITH date_filtered_fixtures AS (
  SELECT DISTINCT id AS fixture_id
  FROM `{FIXTURES_TABLE_ID}`
  WHERE id IS NOT NULL
    AND DATE(start_date) BETWEEN @start_date AND @end_date
    AND UPPER(IFNULL(status, '')) != 'CANCELLED'
),
base AS (
  SELECT o.*
  FROM `{ODDS_TABLE_ID}` o
  INNER JOIN date_filtered_fixtures dff
    ON o.fixture_id = dff.fixture_id
)
SELECT 'market' AS field, COUNT(*) AS matching_rows
FROM base
WHERE LOWER(market) IN UNNEST(@markets)
UNION ALL
SELECT 'market_id' AS field, COUNT(*) AS matching_rows
FROM base
WHERE LOWER(market_id) IN UNNEST(@markets)
UNION ALL
SELECT 'coalesce_market_id_market' AS field, COUNT(*) AS matching_rows
FROM base
WHERE LOWER(COALESCE(NULLIF(market_id, ''), NULLIF(market, ''))) IN UNNEST(@markets)
"""

diagnostic_job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter("markets", "STRING", [m.lower() for m in MARKETS]),
        bigquery.ScalarQueryParameter("start_date", "DATE", START_DATE),
        bigquery.ScalarQueryParameter("end_date", "DATE", END_DATE),
    ]
)

client.query(diagnostic_query, job_config=diagnostic_job_config).result().to_dataframe()


E0000 00:00:1783612141.512730 2444645 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,field,matching_rows
0,market,199962
1,market_id,1240728
2,coalesce_market_id_market,1240728


In [6]:
# Coverage by season_type (tournament name): any available line qualifies a fixture.
season_type_query = f"""
WITH date_filtered_fixtures AS (
  SELECT DISTINCT id AS fixture_id
  FROM `{FIXTURES_TABLE_ID}`
  WHERE id IS NOT NULL
    AND DATE(start_date) BETWEEN @start_date AND @end_date
    AND UPPER(IFNULL(status, '')) != 'CANCELLED'
),
fixture_season_type AS (
  SELECT
    o.fixture_id,
    COALESCE(NULLIF(TRIM(o.season_type), ''), '(missing)') AS season_type
  FROM `{ODDS_TABLE_ID}` o
  INNER JOIN date_filtered_fixtures dff
    ON o.fixture_id = dff.fixture_id
  WHERE o.fixture_id IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY o.fixture_id
    ORDER BY CASE WHEN NULLIF(TRIM(o.season_type), '') IS NULL THEN 1 ELSE 0 END, o.season_type
  ) = 1
),
fixtures_with_any_lines AS (
  SELECT DISTINCT o.fixture_id
  FROM `{ODDS_TABLE_ID}` o
  INNER JOIN date_filtered_fixtures dff
    ON o.fixture_id = dff.fixture_id
  WHERE o.fixture_id IS NOT NULL
    AND IFNULL(o.no_odds, FALSE) = FALSE
)
SELECT
  fst.season_type,
  COUNT(DISTINCT fst.fixture_id) AS total_fixtures,
  COUNT(DISTINCT fwa.fixture_id) AS fixtures_with_lines,
  SAFE_DIVIDE(COUNT(DISTINCT fwa.fixture_id), COUNT(DISTINCT fst.fixture_id)) AS coverage_pct,
  ROUND(100 * SAFE_DIVIDE(COUNT(DISTINCT fwa.fixture_id), COUNT(DISTINCT fst.fixture_id)), 2) AS coverage_pct_100
FROM fixture_season_type fst
LEFT JOIN fixtures_with_any_lines fwa
  ON fst.fixture_id = fwa.fixture_id
GROUP BY 1
ORDER BY total_fixtures DESC, coverage_pct_100 DESC, season_type
"""

season_type_job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("start_date", "DATE", START_DATE),
        bigquery.ScalarQueryParameter("end_date", "DATE", END_DATE),
    ]
)

season_type_coverage_df = client.query(
    season_type_query,
    job_config=season_type_job_config,
).result().to_dataframe()

print(season_type_coverage_df.to_string(index=False))


E0000 00:00:1783612146.165880 2444645 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


                                      season_type  total_fixtures  fixtures_with_lines  coverage_pct  coverage_pct_100
                       French Open, Paris, France             304                  304      1.000000            100.00
            Australian Open, Melbourne, Australia             254                  254      1.000000            100.00
                 Wimbledon, London, Great Britain             254                  254      1.000000            100.00
                           US Open, New York, USA             253                  253      1.000000            100.00
     Wimbledon, London, Great Britain, Qualifying             224                  224      1.000000            100.00
Australian Open, Melbourne, Australia, Qualifying             224                  223      0.995536             99.55
               US Open, New York, USA, Qualifying             224                  222      0.991071             99.11
                                      Rome, Ital